# 08F – Cross-Validation & Stability Analysis

Enterprise notebook to evaluate the stability and consistency of the bankruptcy prediction model across multiple folds.

## Business Objective
Validate that model performance is consistent across different data splits and quantify performance variability.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier

In [ ]:
DATA_PATH='american_bankruptcy_cleaned.csv'

df=pd.read_csv(DATA_PATH)

if 'status_label' in df.columns:
    y=df['status_label'].map({'alive':0,'failed':1})
    X=df.drop(columns=['status_label'])
elif 'target' in df.columns:
    y=df['target']
    X=df.drop(columns=['target'])
else:
    raise ValueError('Target column not found')


In [ ]:
# Match production model configuration
model=RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scores=cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=['accuracy','precision','recall','f1','roc_auc'],
    return_train_score=False,
    n_jobs=-1
)


In [ ]:
results=pd.DataFrame({
    'Fold':range(1,6),
    'Accuracy':scores['test_accuracy'],
    'Precision':scores['test_precision'],
    'Recall':scores['test_recall'],
    'F1':scores['test_f1'],
    'ROC_AUC':scores['test_roc_auc']
})

summary=results.describe().T[['mean','std','min','max']]
results.to_csv('cross_validation_scores.csv',index=False)
summary.to_csv('cross_validation_summary.csv')

display(results)
display(summary)


In [ ]:
plt.figure(figsize=(10,6))
for metric in ['Accuracy','Precision','Recall','F1','ROC_AUC']:
    plt.plot(results['Fold'],results[metric],marker='o',label=metric)

plt.xlabel('Fold')
plt.ylabel('Score')
plt.title('Cross-Validation Stability')
plt.xticks(results['Fold'])
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('cross_validation_stability.png',dpi=300)
plt.show()

## Business Interpretation

- Low standard deviation across folds indicates a stable and reliable model.
- Large variation suggests the model is sensitive to data splits and may require additional tuning or more representative data.
- Consistent ROC-AUC and F1 scores strengthen confidence in production deployment.

## Deliverables

- `cross_validation_scores.csv`
- `cross_validation_summary.csv`
- `cross_validation_stability.png`

This notebook demonstrates that the reported performance is reproducible and not dependent on a single train/test split.